# ER × full HStack1024 — production regression + activity reporting + OOD

This notebook uses all query rows and excludes Test completely. `smiles_tr` is
the endpoint-specific Train+Validation 80%; `data` is every SMILES in
`cleaned_Casestudy.csv`. OOD uses frozen-MinMax-scaled HStack1024 (1024 dimensions). MACCSFingerprint is
not used.


## Controlled workflow

| Step | Stage | Input | Process | Output |
|---:|---|---|---|---|
| 1 | Domain | Raw ER CSV | Deterministic 60/20/20; combine Train+Validation only | `smiles_tr` (80%) |
| 2 | Query | `cleaned_Casestudy.csv` | Use every validated `Smiles` row | `data` |
| 3 | Extraction | `smiles_tr` + `data` | Frozen SMILES/SELFIES/Graph/ECFP extractors | HStack1024 |
| 4 | OOD space | HStack1024 | frozen-MinMax-scaled HStack1024 (1024 dimensions) | `X_train`, `X_test` |
| 5 | Regression | Query feature vector | frozen baseline Ridge | Predicted pIC50 |
| 6 | Activity layer | Train+Validation pIC50 | Professor-style LDA; Positive if pIC50 ≥ 6.0 | Predicted + Probability |
| 7 | OOD | `X_train`, `X_test` | Professor kNN equation for k=3…25 | ADk3…ADk25 |
| 8 | Export | Activity + OOD results | Professor-compatible column order | `IND_Result.csv` |

`Probability` is the LDA posterior probability of class 0 (Positive). It is not
generated by kNN. Test rows never enter the LDA, regression inference, or OOD.


In [ ]:
# Production configuration
RUN_DATASET = 'ER'
RUN_FEATURE_SPACE = 'hstack1024'
RUN_METHOD = None

GITHUB_REPO_URL = "https://github.com/bigmanaschai/RegressionPipeline_hstack1024_FS.git"
GITHUB_REF = "main"
PROJECT_ROOT_OVERRIDE = ""
BASE_ROOT_OVERRIDE = ""
FS_ROOT_OVERRIDE = ""
QUERY_CSV = "/kaggle/input/datasets/manaschaiaonon/ood-regression-arergrpr/cleaned_Casestudy.csv"
QUERY_ID_COLUMN = "ID"
QUERY_SMILES_COLUMN = "Smiles"
OUTPUT_ROOT_OVERRIDE = ""
ACTIVITY_PIC50_THRESHOLD = 6.0
OOD_K_VALUES = tuple(range(3, 26))

assert RUN_DATASET in {"AR", "ER", "GR", "PR"}
assert RUN_FEATURE_SPACE in {"hstack1024", "selectkbest"}
assert (RUN_METHOD in {"mutual_info", "pearson"}) == (RUN_FEATURE_SPACE == "selectkbest")
assert OOD_K_VALUES == tuple(range(3, 26))


In [ ]:
# Resolve or clone the GitHub project without modifying Kaggle Inputs.
from pathlib import Path
import subprocess

EXPECTED_RELATIVE = Path("standard_pipeline/RegressionPipeline_hstack1024_FS")

def find_extension_root(project_root):
    """Support both the monorepo layout and this package's standalone GitHub repo."""
    nested = project_root / EXPECTED_RELATIVE
    if nested.is_dir():
        return nested
    if (
        (project_root / "src" / "hstack1024_fs_pipeline").is_dir()
        and (project_root / "materials" / "hstack1024-fs-extension").is_dir()
        and (project_root / "pyproject.toml").is_file()
    ):
        return project_root
    raise FileNotFoundError(
        f"RegressionPipeline_hstack1024_FS source not found under: {project_root}"
    )

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif (Path.cwd() / EXPECTED_RELATIVE).is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
else:
    if not GITHUB_REPO_URL:
        raise ValueError(
            "Set GITHUB_REPO_URL to the repository pushed from this materials folder, "
            "or set PROJECT_ROOT_OVERRIDE to an existing checkout."
        )
    checkout = Path("/kaggle/working/hstack1024-fs-repository")
    if not checkout.is_dir():
        subprocess.check_call(
            ["git", "clone", "--no-checkout", "--filter=blob:none", GITHUB_REPO_URL, str(checkout)]
        )
    subprocess.check_call(
        ["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", GITHUB_REF]
    )
    subprocess.check_call(
        ["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]
    )
    PROJECT_ROOT = checkout.resolve()

EXTENSION_ROOT = find_extension_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Extension   :", EXTENSION_ROOT)


In [ ]:
# Install only missing optional deep-inference dependencies.
import importlib.util
import os
import subprocess
import sys

# These frozen inference pipelines do not use experiment tracking.
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("WANDB_SILENT", "true")

required_modules = {
    "transformers": "transformers==4.48.3",
    "sentencepiece": "sentencepiece",
    "selfies": "selfies",
    "rdkit": "rdkit",
    "deepchem": "deepchem",
    "torch_geometric": "torch-geometric",
    "skfp": "git+https://github.com/plenoi/scikit-finger-plenoi.git@master",
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from transformers import AutoModel, AutoTokenizer

def pin_remote_code_revision(auto_class):
    original = auto_class.from_pretrained
    if getattr(original, "_hstack_revision_pinned", False):
        return
    def pinned(pretrained_model_name_or_path, *args, **kwargs):
        revision = kwargs.get("revision")
        if revision:
            kwargs.setdefault("code_revision", revision)
        return original(pretrained_model_name_or_path, *args, **kwargs)
    pinned._hstack_revision_pinned = True
    auto_class.from_pretrained = pinned

pin_remote_code_revision(AutoTokenizer)
pin_remote_code_revision(AutoModel)
print("Optional dependencies: OK")


In [ ]:
# Resolve packages and verify the small GitHub material bundle byte-for-byte.
import hashlib
import importlib
import json
import sys

BASE_KAGGLE_ROOT = Path("/kaggle/input/datasets/manaschaiaonon/hstack1024-pipeline-libs")
LOCAL_BASE_ROOT = PROJECT_ROOT / "standard_pipeline/frozen_hstack1024_ridge/kaggle_dataset/hstack1024-pipeline-libs"
BASE_ROOT_CANDIDATE = Path(BASE_ROOT_OVERRIDE).expanduser() if BASE_ROOT_OVERRIDE else (
    BASE_KAGGLE_ROOT if BASE_KAGGLE_ROOT.is_dir() else LOCAL_BASE_ROOT
)

def find_base_artifact_root(candidate):
    """Accept direct or one-level/nested Kaggle uploads of the flat bundle."""
    candidate = Path(candidate)
    direct = candidate / "checkpoints" / "model_smiles_AR.pt"
    if direct.is_file():
        return candidate
    roots = sorted({
        path.parent.parent.resolve()
        for path in candidate.rglob("model_smiles_AR.pt")
        if path.parent.name == "checkpoints"
    })
    if len(roots) == 1:
        return roots[0]
    return candidate

BASE_ROOT = find_base_artifact_root(BASE_ROOT_CANDIDATE)
FS_ROOT = Path(FS_ROOT_OVERRIDE).expanduser() if FS_ROOT_OVERRIDE else (
    EXTENSION_ROOT / "materials/hstack1024-fs-extension"
)
BASE_CODE_SRC = FS_ROOT / "base_src"
FS_SRC = FS_ROOT / "src"
if not BASE_ROOT.is_dir():
    raise FileNotFoundError(f"Attach base Kaggle library; root not found: {BASE_ROOT}")
if not (BASE_CODE_SRC / "hstack1024_pipeline").is_dir():
    raise FileNotFoundError(f"Vendored base source package not found: {BASE_CODE_SRC}")
if not (FS_SRC / "hstack1024_fs_pipeline").is_dir():
    raise FileNotFoundError(f"FS source package not found: {FS_SRC}")

manifest = json.loads((FS_ROOT / "MANIFEST.json").read_text(encoding="utf-8"))
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
for relative, metadata in manifest["files"].items():
    path = FS_ROOT / relative
    if not path.is_file() or sha256(path) != metadata["sha256"]:
        raise RuntimeError(f"FS material missing or stale: {relative}")

for module_name in list(sys.modules):
    if (
        module_name == "hstack1024_pipeline"
        or module_name.startswith("hstack1024_pipeline.")
        or module_name == "hstack1024_fs_pipeline"
        or module_name.startswith("hstack1024_fs_pipeline.")
    ):
        del sys.modules[module_name]
sys.path[:0] = [str(FS_SRC), str(BASE_CODE_SRC)]
importlib.invalidate_caches()

import hstack1024_pipeline
import hstack1024_fs_pipeline

expected_imports = {
    "hstack1024_pipeline": (hstack1024_pipeline, BASE_CODE_SRC / "hstack1024_pipeline"),
    "hstack1024_fs_pipeline": (hstack1024_fs_pipeline, FS_SRC / "hstack1024_fs_pipeline"),
}
for package_name, (package, expected_dir) in expected_imports.items():
    imported_file = getattr(package, "__file__", None)
    if imported_file is None or Path(imported_file).resolve().parent != expected_dir.resolve():
        raise RuntimeError(
            f"Imported {package_name} from {imported_file!r}; expected {expected_dir}"
        )

base_config = importlib.import_module("hstack1024_pipeline.config")
print("Base config    :", base_config.__file__)
print(
    "Artifact dirs :",
    {
        directory: (BASE_ROOT / directory).is_dir()
        for directory in ("checkpoints", "models", "transformers")
    },
)
print("Flat bundle    :", base_config.is_flat_kaggle_bundle(BASE_ROOT))

base_contract = getattr(hstack1024_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if base_contract != "fresh-raw-split-extraction-cpu-v2":
    raise RuntimeError(
        "Incompatible base HStack1024 contract: "
        f"{base_contract!r} from {hstack1024_pipeline.__file__}"
    )
fs_contract = getattr(hstack1024_fs_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if fs_contract != "fresh-raw-split-extraction-selectkbest-cpu-v1":
    raise RuntimeError(
        "Incompatible HStack1024 FS contract: "
        f"{fs_contract!r} from {hstack1024_fs_pipeline.__file__}"
    )
print("Base root     :", BASE_ROOT)
print("FS materials  :", FS_ROOT)
print("Inference device: CPU")


# Require the production OOD runtime from the checked-out, manifest-verified source.
ood_source = FS_SRC / "hstack1024_fs_pipeline" / "ood.py"
if not ood_source.is_file():
    raise RuntimeError(
        "Production OOD runtime is absent from this checkout: "
        f"{ood_source}. Restart the session and Run All with the latest GITHUB_REF."
    )
ood_module = importlib.import_module("hstack1024_fs_pipeline.ood")
if Path(ood_module.__file__).resolve() != ood_source.resolve():
    raise RuntimeError(
        f"Imported production OOD runtime from {ood_module.__file__!r}; "
        f"expected {ood_source}"
    )
print("Production OOD runtime:", ood_module.__file__)


In [ ]:
# Resolve only the production inputs and frozen artifacts.
from hstack1024_pipeline.config import (
    FAMILY_ORDER, checkpoint_path, fingerprint_transformer_path, raw_csv_path,
    ridge_path, scaler_path,
)
from hstack1024_fs_pipeline.config import bundle_path, get_variant
from hstack1024_fs_pipeline.ood import load_query_compounds

raw_csv = raw_csv_path(RUN_DATASET, BASE_ROOT)
query_csv = Path(QUERY_CSV).expanduser()
if not raw_csv.is_file():
    raise FileNotFoundError(raw_csv)
if not query_csv.is_file():
    raise FileNotFoundError(f"Production query CSV not found: {query_csv}")
query_preview = load_query_compounds(
    query_csv, id_column=QUERY_ID_COLUMN, smiles_column=QUERY_SMILES_COLUMN
)
for family in FAMILY_ORDER:
    artifact = checkpoint_path(RUN_DATASET, family, BASE_ROOT)
    if not artifact.is_file():
        raise FileNotFoundError(artifact)
transformer = fingerprint_transformer_path(RUN_DATASET, BASE_ROOT)
if not transformer.is_file():
    raise FileNotFoundError(transformer)

if RUN_FEATURE_SPACE == "selectkbest":
    model_artifact = bundle_path(RUN_DATASET, RUN_METHOD, fs_root=FS_ROOT)
    spec = get_variant(RUN_DATASET, RUN_METHOD)
    print("Variant:", spec.variant_id, spec.model_class, f"FS_k={spec.selected_k}")
else:
    model_artifact = ridge_path(RUN_DATASET, BASE_ROOT)
    normalizer_artifact = scaler_path(RUN_DATASET, BASE_ROOT)
    if not normalizer_artifact.is_file():
        raise FileNotFoundError(normalizer_artifact)
if not model_artifact.is_file():
    raise FileNotFoundError(model_artifact)

print("Query CSV  :", query_csv)
print("Query rows :", len(query_preview))
print("Feature OOD:", RUN_FEATURE_SPACE)
print("Inputs: OK")


In [ ]:
# Run regression, professor-style LDA activity reporting, and OOD.
from hstack1024_fs_pipeline.ood import run_production_ood_dataset

variant_name = (
    f"{RUN_DATASET}_{RUN_METHOD}"
    if RUN_FEATURE_SPACE == "selectkbest"
    else f"{RUN_DATASET}_hstack1024"
)
default_output = Path("/kaggle/working") / f"production_ood_{variant_name}"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser() if OUTPUT_ROOT_OVERRIDE else default_output

ood_report = run_production_ood_dataset(
    RUN_DATASET,
    query_csv,
    OUTPUT_ROOT,
    feature_space=RUN_FEATURE_SPACE,
    method=RUN_METHOD,
    base_root=BASE_ROOT,
    fs_root=FS_ROOT,
    repo_root=PROJECT_ROOT,
    raw_csv=raw_csv,
    id_column=QUERY_ID_COLUMN,
    smiles_column=QUERY_SMILES_COLUMN,
    activity_threshold=ACTIVITY_PIC50_THRESHOLD,
)
print("Production output:", OUTPUT_ROOT)
ood_report


In [ ]:
# Plot IND coverage for all mandatory k values.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.plot(ood_report["k"], ood_report["IND_Coverage"], marker="o")
ax.set(title="New-compound applicability-domain coverage", xlabel="k", ylabel="IND coverage")
ax.set_xticks(range(3, 26, 2))
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.25)
figure_path = OUTPUT_ROOT / "ood_coverage_diagnostics.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Coverage figure:", figure_path)


In [ ]:
# Verify and display the professor-compatible report.
import pandas as pd

result_files = sorted(OUTPUT_ROOT.rglob("IND_Result.csv"))
if len(result_files) != 1:
    raise RuntimeError(f"Expected exactly one IND_Result.csv, found {len(result_files)}")
detailed_files = sorted(OUTPUT_ROOT.rglob("production_predictions_ood.csv"))
if len(detailed_files) != 1:
    raise RuntimeError(
        f"Expected exactly one production_predictions_ood.csv, found {len(detailed_files)}"
    )
result = pd.read_csv(result_files[0])
expected = ["Unnamed: 0", "Smiles", "Predicted", "Probability"] + [f"ADk{k}" for k in range(3, 26)]
if result.columns.tolist() != expected:
    raise AssertionError(f"IND_Result.csv columns changed: {result.columns.tolist()}")
if not set(result["Predicted"].unique()).issubset({"Positive", "Negative"}):
    raise AssertionError("Unexpected Predicted label")
if not result["Probability"].between(0.0, 1.0).all():
    raise AssertionError("Probability must be in [0,1]")
print("IND result:", result_files[0])
print("Rows:", len(result))
display(result.head())
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_ROOT))


## Interpretation

- `Predicted` is Positive/Negative from the newly fitted LDA activity layer.
- `Probability` is the LDA probability for Positive, not a kNN probability.
- `ADk3`…`ADk25` independently report IND/OOD using the professor's equation.
- `Predicted_pIC50` remains available in `production_predictions_ood.csv`.
- OOD status is an applicability warning and does not replace activity class.
